# 08 Visual Seaborn

Seaborn-centric visuals over sessions and clustering/model outputs from DuckDB.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from utils.duckdb_utils import query_df, table_exists
from utils.paths import get_db_paths, resolve_workspace

sns.set_theme(style='whitegrid', context='talk')

workspace = resolve_workspace(Path.cwd())
input_db, output_db = get_db_paths(workspace)

sessions = query_df(input_db, 'SELECT title, day, start_time_24h, track FROM sessions_in_preprocessed')
sessions['hour'] = pd.to_numeric(sessions['start_time_24h'].str.slice(0, 2), errors='coerce')
            


In [ ]:
plot_df = sessions.dropna(subset=['hour'])
plt.figure(figsize=(12, 6))
sns.violinplot(data=plot_df, x='day', y='hour', inner='quartile', cut=0, palette='Set2')
sns.swarmplot(data=plot_df.sample(min(len(plot_df), 120), random_state=42), x='day', y='hour', color='black', alpha=0.45, size=3)
plt.title('Session Start-Time Spread by Day')
plt.tight_layout()
plt.show()

In [ ]:
top_tracks = sessions['track'].value_counts().head(12).index
hm = sessions[sessions['track'].isin(top_tracks)].pivot_table(index='day', columns='track', values='title', aggfunc='count', fill_value=0)
plt.figure(figsize=(14, 5))
sns.heatmap(hm, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Schedule Density by Day and Top Tracks')
plt.tight_layout()
plt.show()

In [ ]:
if not output_db.exists() or not table_exists(output_db, 'clustering_assignments'):
    print('Run notebook 04 to populate clustering output tables.')
else:
    cl = query_df(
        output_db,
        'SELECT CAST(cluster_id AS VARCHAR) AS cluster_id, day, COUNT(*) AS n FROM clustering_assignments GROUP BY 1,2'
    )
    p = cl.pivot(index='day', columns='cluster_id', values='n').fillna(0)
    plt.figure(figsize=(10, 5))
    sns.heatmap(p, annot=True, fmt='.0f', cmap='rocket_r')
    plt.title('Cluster Distribution by Day')
    plt.tight_layout()
    plt.show()
            
